This notebook demonstrates a simple linear regression analysis using Python to model Salary based on Years of Experience. It includes annotated regression plots and detailed diagnostic checks (residuals, normality, and influence metrics).

In [ ]:
import pandas as pd

dataset = pd.read_csv("regression_data.csv")
dataset.head()

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(dataset["YearsExperience"], dataset["Salary"], color="red")
plt.title("Salary vs Experience")
plt.xlabel("Years of Experience")
plt.ylabel("Salary")
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(dataset[["YearsExperience"]], dataset[["Salary"]])

In [ ]:
plt.scatter(dataset["YearsExperience"], dataset["Salary"], color="red")
plt.plot(
    dataset["YearsExperience"],
    model.predict(dataset[["YearsExperience"]]),
    color="blue",
)
plt.title("Salary vs Experience")
plt.xlabel("Years of Experience")
plt.ylabel("Salary")
plt.show()

In [ ]:
r_squared = model.score(dataset[["YearsExperience"]], dataset[["Salary"]])
print(f"R-squared: {r_squared:.4f}")

In [ ]:
import numpy as np
from scipy import stats

x = dataset[["YearsExperience"]].values
y = dataset[["Salary"]].values.ravel()
y_pred = model.predict(dataset[["YearsExperience"]]).ravel()
residuals = y - y_pred
r_squared = model.score(dataset[["YearsExperience"]], dataset[["Salary"]])
n = len(y)
adj_r_squared = 1 - (1 - r_squared) * (n - 1) / (n - 2)
intercept = float(np.atleast_1d(model.intercept_)[0])
slope = float(np.atleast_1d(model.coef_.ravel())[0])
rmse = np.sqrt(np.mean(residuals**2))

x_line_flat = np.linspace(x.min(), x.max(), 100)
x_line = x_line_flat.reshape(-1, 1)
y_line = model.predict(x_line).ravel()
residual_std = np.std(residuals, ddof=2)
t_crit = stats.t.ppf(0.975, n - 2)
x_mean = x.mean()
x_ss = np.sum((x - x_mean) ** 2)
ci_margin = t_crit * residual_std * np.sqrt(1 / n + (x_line_flat - x_mean) ** 2 / x_ss)
y_upper = y_line + ci_margin
y_lower = y_line - ci_margin

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(dataset["YearsExperience"], dataset["Salary"], color="red", alpha=0.8, label="Observed")
ax.plot(x_line_flat, y_line, color="blue", linewidth=2, label="Fitted line")
ax.fill_between(x_line_flat, y_lower, y_upper, color="blue", alpha=0.15, label="95% CI")
annotation = (
    f"Salary = {intercept:,.0f} + {slope:,.0f} * YearsExperience\n"
    f"R² = {r_squared:.4f}\nAdj. R² = {adj_r_squared:.4f}\nRMSE = {rmse:,.0f}"
)
ax.annotate(annotation, xy=(0.05, 0.95), xycoords="axes fraction", va="top",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.85))
ax.set_title("Salary vs Experience — Annotated Regression")
ax.set_xlabel("Years of Experience")
ax.set_ylabel("Salary")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("regression_plot_python.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes[0, 0].scatter(y_pred, residuals, color="steelblue")
axes[0, 0].axhline(0, color="gray", linestyle="--")
axes[0, 0].set_title("Residuals vs Fitted")
axes[0, 0].set_xlabel("Fitted values")
axes[0, 0].set_ylabel("Residuals")

stats.probplot(residuals, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title("Normal Q-Q Plot")

standardized_residuals = residuals / residual_std
axes[1, 0].scatter(y_pred, np.sqrt(np.abs(standardized_residuals)), color="darkorange")
axes[1, 0].set_title("Scale-Location")
axes[1, 0].set_xlabel("Fitted values")
axes[1, 0].set_ylabel("√|Standardized residuals|")

axes[1, 1].hist(residuals, bins=6, color="seagreen", edgecolor="white")
axes[1, 1].set_title("Residual Distribution")
axes[1, 1].set_xlabel("Residual")
axes[1, 1].set_ylabel("Count")

fig.suptitle("Regression Diagnostic Plots")
fig.tight_layout()
plt.savefig("regression_diagnostics_python.png", dpi=150)
plt.show()

leverage = (1 / n) + ((x - x.mean()) ** 2 / np.sum((x - x.mean()) ** 2)).ravel()
cooks_d = (residuals**2 / residual_std**2) * (leverage / (1 - leverage) ** 2)
print(f"Residual mean: {residuals.mean():,.2f}")
print(f"Residual std: {residual_std:,.2f}")
print(f"Max |residual|: {np.max(np.abs(residuals)):,.2f}")
print(f"Max Cook's distance: {cooks_d.max():.4f}")